# Task 6 — Idempotent Replay Verification

## Approach and reasoning

The lab asks us to modify one Python file, reprocess **only that file**, and show
that all three systems reflect the update without duplication.

Idempotency is enforced at three independent layers, which is deliberate — any
one of them failing would otherwise corrupt the graph silently:

| Layer | Mechanism |
|---|---|
| Parser | structural, line-independent identifiers |
| Neo4j sink | `MERGE` on the stable id (never `CREATE`) |
| Spark → Mongo | upsert on `file_id` + offset checkpoint |

### The problem the three layers do *not* solve

`MERGE` guarantees that re-emitted elements are updated rather than duplicated.
It says nothing about elements that are **no longer emitted** — when an edit
deletes a function, that function's nodes are simply never sent again, so they
linger in Neo4j as orphans.

We solve this with a **generation sweep**. Every element carries its file's
sha256 as `file_hash`. After reprocessing a file we delete elements of that file
whose `file_hash` differs from the current one. The sweep is scoped to a single
`file_id`, so it can never touch another file's subgraph.

### Step 1 — source-level proof, before touching any database

In [ ]:
import os, sys
os.chdir("..")
print("cwd:", os.getcwd())

In [ ]:
!python src/replay/verify_replay.py ./optimum

The line to read carefully is **`nodes surviving the edit`**. The edit prepends a
comment, shifting every line number in the file, yet the original nodes keep
their identifiers. A line-based identifier scheme would report zero survivors and
duplicate the entire file downstream.

### Step 2 — record the "before" state of both databases

In [ ]:
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH (n:CpgNode {rel_path:'optimum/version.py'}) RETURN count(n) AS nodes_before"
!docker exec mongodb mongosh --quiet cpg --eval \
  "db.file_metadata.countDocuments({rel_path:'optimum/version.py'})"

### Step 3 — actually modify the file

In [ ]:
target = "optimum/optimum/version.py"
src = open(target).read()
if "_lab_replay_marker" not in src:
    open(target, "w").write(
        "# lab04 replay edit: shifts every line number below\n"
        + src
        + "\n\ndef _lab_replay_marker(x):\n    y = x + 1\n    return y\n"
    )
print(open(target).read()[:300])

### Step 4 — refresh the manifest so the new content hash is picked up

In [ ]:
!python src/discovery/discover_files.py --repo ./optimum --out reports/file_manifest.json

### Step 5 — reprocess ONLY that file

In [ ]:
!python src/parser/parser_service.py --manifest reports/file_manifest.json \
    --repo ./optimum --only optimum/version.py --bootstrap localhost:9092

### Step 6 — run the generation sweep to remove stale nodes

In [ ]:
!python src/replay/sweep.py --repo ./optimum --rel-path optimum/version.py \
    --uri bolt://localhost:7687 --user neo4j --password password

### Step 7 — verify Neo4j: updated, no duplicates

In [ ]:
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH (n:CpgNode {rel_path:'optimum/version.py'}) RETURN count(n) AS nodes_after"
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH (n:CpgNode) WITH n.id AS id, count(*) AS c WHERE c > 1 RETURN id, c"

### Step 8 — verify MongoDB: document updated, still exactly one

In [ ]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "db.file_metadata.countDocuments({rel_path:'optimum/version.py'})"
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.findOne({rel_path:'optimum/version.py'},{rel_path:1,file_hash:1,num_ast_nodes:1,num_functions:1}))"

### Step 9 — verify the Spark checkpoint skipped unchanged offsets

Paste the streaming log from the running job. The evidence is that the batch
triggered by the replay contains **1** document, not 59 — Spark resumed from the
committed offset and read only the newly produced message.

```
batch 1: upserted 1 metadata docs
```

## Reflection

*(Replace this with what actually happened on your machine.)*

**What worked.** The structural identifier design paid off exactly here. All
original nodes survived a full line shift, so Neo4j saw an update rather than a
fresh copy of the file.

**What we nearly missed.** `MERGE` alone is not enough for true idempotency.
Elements removed by an edit are never re-emitted and therefore never touched by
`MERGE` — they simply accumulate. We only noticed when the node count for the
edited file grew instead of staying stable. The generation sweep closed the gap.

**Trade-off we accepted.** The sweep is a separate step after the connector has
drained, rather than something the connector does itself. A Cypher sink statement
cannot know that a file is "finished", so a delete-then-write strategy inside the
connector would race with in-flight messages. Running the sweep afterwards is
slower but correct.